<a href="https://colab.research.google.com/github/elhamod/BA305_Fall_2026/blob/main/Session%2015%20-%20Logistic%20Regression/lab7_Logistic_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Lab 7: Logistic Regression : Identifying Good System Administrators**

![picture](https://drive.google.com/uc?export=view&id=1Cha59qNclnjd-coOz1WWUDnKOuVNOaiK)

Questrom & Company (QC), a consulting firm specialized in information systems, is studying the roles played by **experience** and **training** in a system administrator’s ability to **complete a set of tasks in a specified amount of time**. In particular, the firm needs to be able to discriminate between administrators who usually are able to complete given tasks within a specified time and those who are not. Data are collected on the performance of 75 randomly selected administrators. They are stored in the file _SystemAdministrators.csv_.

The variable **'experience'** measures months of full-time system administrator experience, while **'training'** measures the number of relevant training credits. The outcome variable **'completed task'** is either Yes or No, according to whether or not the administrator completed the tasks.

Note, this is problem 10.2 from the DMBA book.

## Setup

In [ ]:
# Load the libraries used throughout this lab.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split


## Pre-processing

In [ ]:
# Load the 75 system administrators and preview the first rows.
admin_df = pd.read_csv('https://raw.githubusercontent.com/AnalyticsArmory/data/main/SystemAdministrators.csv')
admin_df.head()


In [ ]:
# Replace the space in 'Completed task' with an underscore so it can be typed as admin_df.Completed_task.
admin_df.columns = [c.replace(' ', '_') for c in admin_df.columns]
admin_df.columns


In [ ]:
# Turn the Yes/No outcome into 1/0, which is what the model needs.
admin_df['Completed_task'] = admin_df['Completed_task'].map({'Yes': 1, 'No': 0})
admin_df.head()


## Visualization

In [ ]:
# Scatter plot of Training vs. Experience, colored by whether the task was completed.
fig, ax = plt.subplots()

subset0 = admin_df.loc[admin_df['Completed_task'] == 0]
ax.scatter(subset0.Training, subset0.Experience, marker='x', label='No', color='red')

subset1 = admin_df.loc[admin_df['Completed_task'] == 1]
ax.scatter(subset1.Training, subset1.Experience, marker='o', label='Yes', color='green')

plt.xlabel('Training')
plt.ylabel('Experience')
ax.legend();


**Question 1.** Looking at the scatter plot, which predictor, Experience or Training, seems more useful for telling apart administrators who complete the task from those who do not?

**Answer**

*Provide your answer here*

In [ ]:
# Pick the two predictors (X) and the outcome (y) from the table.
X = admin_df[['Experience', 'Training']]
y = admin_df['Completed_task']


In [ ]:
# Hold out 40% of administrators as a test set, keeping the share of Yes/No the same in both halves.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=1, stratify=y)


**Question 2.** What does the `stratify=y` option do, and why does it matter with only 75 administrators?

**Answer**

*Provide your answer here*

## Run the Logistic Regression model

In [ ]:
# Fit a plain logistic regression (no penalty) on the training data.
lr = LogisticRegression(penalty=None, solver='lbfgs')
lr.fit(X_train, y_train)


In [ ]:
# Predict Yes/No (1/0) for the test administrators, using the default 0.5 cutoff.
y_pred = lr.predict(X_test)


In [ ]:
# Confusion matrix: rows are the actual outcome (0, 1), columns the predicted one.
confusion_matrix(y_test, y_pred)


In [ ]:
# Share of test administrators classified correctly.
accuracy_score(y_test, y_pred)


### Changing the cutoff

`predict` labels an administrator 1 whenever the predicted probability is above 0.5. To use a different cutoff, we ask the model for the probabilities themselves and apply the cutoff ourselves.

In [ ]:
# Put each test administrator's data, actual outcome, predicted probability and 0.5-cutoff prediction side by side.
p_pred = lr.predict_proba(X_test)[:, 1]

results_df = X_test.copy()
results_df['Actual'] = y_test
results_df['Probability'] = np.round(p_pred, 3)
results_df['Predicted_0.5'] = y_pred
results_df


In [ ]:
# Label an administrator 1 whenever the probability is above the new cutoff, and add it as a column.
cutoff = 0.25
y_pred_new = [1 if p > cutoff else 0 for p in p_pred]

results_df['Predicted_0.25'] = y_pred_new
results_df.sort_values('Probability', ascending=False)   # highest probability first


In [ ]:
# Confusion matrix with the new cutoff.
confusion_matrix(y_test, y_pred_new)


In [ ]:
# Accuracy with the new cutoff.
accuracy_score(y_test, y_pred_new)


**Question 3.** Compare the confusion matrices at the 0.5 and 0.25 cutoffs. What changed, and when would QC prefer the lower cutoff?

**Answer**

*Provide your answer here*

## Interpreting results

In [ ]:
# The fitted intercept and the coefficient of each predictor (on the logit scale).
lr_int = lr.intercept_[0]
lr_coef = lr.coef_[0]

print('intercept ', lr_int)
print(pd.DataFrame({'coeff': lr_coef}, index=X.columns))


In [ ]:
# Exponentiate the coefficients to read them on the odds scale.
print('Experience:', np.exp(lr_coef[0]))
print('Training:  ', np.exp(lr_coef[1]))


**Question 4.** Interpret the two exponentiated coefficients above. What happens to the odds of completing the task on time when Experience goes up by one month? When Training goes up by one credit?

**Answer**

*Provide your answer here*

### From a new administrator's profile to a probability

Logistic regression gives us the **logit** (log-odds) as a straight line in the predictors. Exponentiating it gives the **odds**, and odds / (1 + odds) gives the **probability**. The next cell walks through those three steps by hand.

In [ ]:
# A new administrator with 9 months of Experience and 6 Training credits.
na_exp = 9
na_train = 6

logit = lr_int + lr_coef[0] * na_exp + lr_coef[1] * na_train
print("logit: ", logit)

odds = np.exp(logit)
print("odds: ", odds)

prob = odds / (1 + odds)
print("prob: ", prob)


Now give the same administrator one more month of Experience. Since the odds ratio for Experience is about 3.26, the odds should grow from 1.31 to about 1.31 × 3.26 ≈ 4.26. Let's check.

In [ ]:
# The same administrator with one extra month of Experience (10 instead of 9).
na_exp = 10
na_train = 6

logit = lr_int + lr_coef[0] * na_exp + lr_coef[1] * na_train
print("logit: ", logit)

odds = np.exp(logit)
print("odds: ", odds)

prob = odds / (1 + odds)
print("prob: ", prob)


## Optional: Logistic Regression with statsmodels (to get p-values)

*This section is optional.*

scikit-learn does not report p-values. The `statsmodels` package fits the same model and prints a full statistical summary.

In [ ]:
# statsmodels does not add an intercept on its own, so add a constant column first.
X_train_with_constant = sm.add_constant(X_train)

logit_reg = sm.Logit(y_train, X_train_with_constant).fit()
print(logit_reg.summary())


**Question 5.** Using the `P>|z|` column of the summary, which predictors are statistically significant at the 5% level? Does this agree with your answer to Question 1?

**Answer**

*Provide your answer here*